In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\torchani\aev.py:16: UserWarning: cuaev not installed
  warnings.warn("cuaev not installed")
C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\torchani\__init__.py:39: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rl = 15000
model = PPO.load(f"RLmodels/ppo_rm_{rl}.zip", device="cpu")
model

In [3]:
fragment_pool = pd.read_csv("source/SMILES_list_100.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=2,
                          max_k=3,
                          max_steps=6,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [4]:
df = generate_SMILES(model, eval_env, n_episodes=50)
df.to_csv(f"generated_OPSs/rl_candidates_{rl}.csv", index=False)
df

episode 0 done.
episode 1 done.
episode 2 done.
episode 3 done.
episode 4 done.
episode 5 done.
episode 6 done.
episode 7 done.
episode 8 done.
episode 9 done.
episode 10 done.
episode 11 done.
episode 12 done.
episode 13 done.
episode 14 done.
episode 15 done.
episode 16 done.
episode 17 done.
episode 18 done.
episode 19 done.
episode 20 done.
episode 21 done.
episode 22 done.
episode 23 done.
episode 24 done.
episode 25 done.
episode 26 done.
episode 27 done.
episode 28 done.
episode 29 done.
episode 30 done.
episode 31 done.
episode 32 done.
episode 33 done.
episode 34 done.
episode 35 done.
episode 36 done.
episode 37 done.
episode 38 done.
episode 39 done.
episode 40 done.
episode 41 done.
episode 42 done.
episode 43 done.
episode 44 done.
episode 45 done.
episode 46 done.
episode 47 done.
episode 48 done.
episode 49 done.


,episode,smiles,selected_fragments,reward
0,0,O=C1c2ccccc2-c2ccc(-c3ccc4c(c3)c3ccccc3n4-c3cc...,"(O=C1c2ccccc2-c2ccccc21, c1ccc2c(c1)Nc1ccccc12...",8.138302
1,1,c1cnnc(-n2c3ccccc3c3cc(-n4c5ccccc5c5ccccc54)cc...,"(c1ccc2c(c1)Nc1ccccc12, c1ccc2c(c1)Nc1ccccc12,...",3.810970
2,2,O=[N+]([O-])c1ccc(-c2ccc(-c3ccc([N+](=O)[O-])c...,"(O=[N+]([O-])c1ccccc1, O=[N+]([O-])c1ccccc1, O...",2.008014
3,3,N#Cc1nc(-c2ccc([N+](=O)[O-])cc2)cc(-c2ccc([N+]...,"(N#Cc1ncccn1, O=[N+]([O-])c1ccccc1, O=[N+]([O-...",1.733444
4,4,O=C1c2ccccc2-c2c1cc(-c1ccc([N+](=O)[O-])cc1)cc...,"(O=C1c2ccccc2-c2ccccc21, O=[N+]([O-])c1ccccc1,...",5.240239
5,5,O=[N+]([O-])c1ccc(-c2ccc(-c3ccc([N+](=O)[O-])c...,"(c1ccnnc1, O=[N+]([O-])c1ccccc1, O=[N+]([O-])c...",6.885588
6,6,O=[N+]([O-])c1ccc(-c2ccc(-n3c4ccccc4c4ccccc43)...,"(c1ccc2c(c1)Nc1ccccc12, O=[N+]([O-])c1ccccc1, ...",21.307104
7,7,O=[N+]([O-])c1ccc(-c2ccc(-c3ccc([N+](=O)[O-])c...,"(O=[N+]([O-])c1ccccc1, O=[N+]([O-])c1ccccc1, O...",2.008014
8,8,O=[N+]([O-])c1ccc(-c2ccc(-c3ccc([N+](=O)[O-])c...,"(c1ccnnc1, O=[N+]([O-])c1ccccc1, O=[N+]([O-])c...",6.885588
9,9,O=[N+]([O-])c1ccc(-c2ccc(-n3c4ccccc4c4ccccc43)...,"(c1ccc2c(c1)Nc1ccccc12, O=[N+]([O-])c1ccccc1, ...",21.307104
